# 08 — Routing & Query Construction (Logical, Semantic & Metadata Filters)

**By the end of this notebook, you can route queries cleanly across multiple document collections and translate questions into strict metadata filters.**

### Learning Objectives
- Build Logical Routing using Pydantic schemas and `.with_structured_output()`.
- Build zero-LLM Semantic Routing using fast embedding cosine similarity.
- Construct explicit vector-store metadata filter predicates from query keywords.

### Prerequisites
- Notebooks 00–07

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Traffic Cop Architecture: Routing Before Searching

Imagine walking into a hospital asking *"Where do I pay my invoice?"* If the receptionist sends you to the surgery ward instead of Billing, you waste time and risk wrong advice.

**Query Routing** is an AI receptionist:
1. **Logical Routing**: Uses a structured Pydantic schema (`datasource: Literal['handbook', 'faq', 'changelog', 'safety']`) so the LLM picks an exact catalog bucket.
2. **Semantic Routing**: Zero extra LLM cost! Embeds domain anchor descriptions (*"HR, pets, stipends"* vs *"errors, ROS2, latency"*), compares cosine similarity against the question, and routes instantaneously!
3. **Query Construction**: Converts phrases (*"only look in changelog"*) into structured metadata filters.

```
+---> [company_handbook] (HR, Travel, Pricing)
User Query --> [Router] -+---> [product_faq]      (Errors, ROS2, SLAs)
                         +---> [recent_changelog] (Oct 2026 updates, deprecations)
                         +---> [safety_brief]     (Perimeters, E-Stops)
```

## 3. Step-by-Step Build

### Step 1: Define a Pydantic Logical Router with `.with_structured_output()`
Let's define a strict Pydantic `RouteQuery` schema restricting choices to our four corpus domains and invoke the structured router.

In [2]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    datasource: Literal["company_handbook", "product_faq", "recent_changelog", "safety_brief"] = Field(
        description="Pick the single best document store for the question."
    )

router = llm.with_structured_output(RouteQuery)
for q in ["How do I resolve ERR_QUANTUM_TIMEOUT?", "When are pets allowed in Zurich?", "Is REST API v1 deprecated?"]:
    print(f"Question: {q[:45]:<45} -> Route: {router.invoke(q).datasource}")
        

Question: How do I resolve ERR_QUANTUM_TIMEOUT?         -> Route: product_faq
Question: When are pets allowed in Zurich?              -> Route: company_handbook
Question: Is REST API v1 deprecated?                    -> Route: recent_changelog


> **What just happened?**
> Pydantic validation guarantees the router NEVER invents an unknown collection name.

### Step 2: Build a Zero-LLM Semantic Router Using Embeddings
If you want sub-5ms routing with ZERO LLM token cost, embed descriptive anchor templates for each domain and pick the highest cosine score.

In [3]:
import numpy as np
route_descriptions = {
    "company_handbook": "Employee travel meals, learning stipend, Zurich pet policy, Starter Professional Enterprise plan monthly prices.",
    "product_faq": "Error codes ERR_QUANTUM_TIMEOUT port 9872, ROS2 distributions, ZeroGravity failover uptime SLA.",
    "recent_changelog": "October 2026 release notes, HyperSync 42% latency, Dr. Elina Vance, REST API v1 shutdown.",
}
route_vecs = {k: np.array(embeddings.embed_query(v)) for k, v in route_descriptions.items()}
print("Embedded", len(route_vecs), "semantic route anchors cleanly.")
        

Embedded 3 semantic route anchors cleanly.


> **What just happened?**
> Anchor route vectors only need to be computed once at startup.

### Step 3: Classify Queries via Cosine Similarity
Let's test `semantic_route(query)` on sample questions and confirm fast, accurate classification without calling the LLM.

In [4]:
def semantic_route(query: str) -> str:
    qv = np.array(embeddings.embed_query(query))
    return max(route_vecs, key=lambda name: float(np.dot(qv, route_vecs[name])))

for q in ["What port does QuantumCache error require?", "How much is the daily meal allowance?"]:
    print(f"{q:<48} -> {semantic_route(q)}")
        

What port does QuantumCache error require?       -> product_faq
How much is the daily meal allowance?            -> company_handbook


> **What just happened?**
> Semantic routing routes cleanly using vector geometry alone!

### Step 4: Query Construction — Converting Intent into Vector Filter Predicates
Now let's combine routing with filtered retrieval on an indexed `InMemoryVectorStore` so wrong sources are excluded completely.

In [5]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=340, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
target_src = semantic_route("Tell me about REST API v1 deprecation date") + (".md" if "changelog" in semantic_route("Tell me about REST API v1 deprecation date") else ".txt")
filtered_docs = store.similarity_search("REST API v1", k=2, filter=lambda d: "changelog" in d.metadata.get("source", ""))
print("Filtered match source:", [d.metadata["source"] for d in filtered_docs])
        

Filtered match source: ['recent_changelog.md', 'recent_changelog.md']


> **What just happened?**
> Combining routing with `filter=...` shrinks the search pool and eliminates cross-document confusion.

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Route a safety perimeter query** — *Hint:* Test both `router.invoke(...)` and `semantic_route(...)` on `physical safety perimeter E-Stop`.
2. **Extend `RouteQuery`** — *Hint:* Add a helper mapping datasource strings directly to filenames.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
src_map = {
    "company_handbook": "company_handbook.txt",
    "product_faq": "product_faq.md",
    "recent_changelog": "recent_changelog.md",
    "safety_brief": "policy_brief.pdf",
}
choice = router.invoke("What is the physical safety perimeter during AMR commissioning?").datasource
print("Mapped filename:", src_map[choice])
        

Mapped filename: policy_brief.pdf


## 5. Common Mistakes
- Using brittle `if 'error' in q` keyword rules for routing without a semantic/structured fallback — synonyms will slip through.
- Giving the router ambiguous overlapping bucket descriptions.
- Forgetting a general/multi-doc fallback route when a user asks a cross-file comparison question.

## 6. Recap
- Logical routing enforces exact schema outputs via Pydantic `.with_structured_output()`.
- Semantic routing eliminates LLM latency by matching query vectors directly to domain anchor descriptions.
- Query construction translates user intent into structured metadata `filter` predicates.

## 7. Next Notebook
Next up (`09_advanced_indexing.ipynb`): Separate what you *search* from what you *hand to the LLM* using Multi-Vector summaries and Parent-Child indexing!

## 8. Video Chapter Outline
1. **Why Route Queries First?** — Preventing noisy cross-collection hits.
2. **Logical Routing with Pydantic** — Structured classification schemas.
3. **Zero-LLM Semantic Routing** — Embedding anchor similarity in 5 lines.
4. **Metadata Query Construction** — Filtering the vector store automatically.
5. **Recap & Multi-Doc Fallback Warning** — Handling cross-document questions safely.